# 📈 Calories Burned & BMI Prediction
## Machine Learning Project — Wael
### Objective: Predict Calories Burned and BMI using Linear Regression, Random Forest and XGBoost

**Features used:** 9 meaningful activity & physiological features  
**Targets:** `Calories_Burned` (regression) + `BMI_calc` (regression)  
**Key finding:** Data leakage identified and fixed — only clean features used

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Imports successful")

## 2. Load & Clean Dataset

In [ ]:
df = pd.read_csv('df_clean.csv')
print(f'Shape: {df.shape}')

# Fix known data quality issues
df['sugar_g']        = df['sugar_g'].clip(lower=0)
df['cholesterol_mg'] = df['cholesterol_mg'].clip(lower=0)

print(f'Nulls: {df.isnull().sum().sum()}')
df.head(3)

## 3. Feature Selection & Target Definition

### Why only 9 features?

The dataset has 54 columns but most are irrelevant or cause **data leakage**.

**Leaky columns excluded:**
- `cal_balance` = Calories - Calories_Burned → directly derives the target
- `Burns_Calories_Bin` → encoded from Calories_Burned
- `BMI` → mathematically identical to `BMI_calc` (Weight/Height²)
- `Weight (kg)` and `Height (m)` → define BMI_calc mathematically

**9 clean features selected:**

| Feature | Why it's relevant |
|---------|------------------|
| Age | Metabolism slows with age |
| Fat_Percentage | Body composition affects burn rate |
| Session_Duration | More time = more calories |
| Avg_BPM | Heart rate reflects exercise intensity |
| Resting_BPM | Baseline cardiovascular fitness |
| Workout_Frequency | Training consistency |
| Water_Intake | Hydration affects performance |
| Experience_Level | Trained athletes burn differently |
| Workout_Type | Different exercises have different burn rates |

In [ ]:
deploy_features = [
    'Age', 'Fat_Percentage', 'Session_Duration (hours)',
    'Avg_BPM', 'Resting_BPM', 'Workout_Frequency (days/week)',
    'Water_Intake (liters)', 'Experience_Level', 'Workout_Type'
]

# Encode Workout_Type
df_model = df.copy()
df_model['Workout_Type'] = df_model['Workout_Type'].map(
    {'Cardio': 0, 'HIIT': 1, 'Strength': 2, 'Yoga': 3}
)

X   = df_model[deploy_features]
y_cal = df_model['Calories_Burned']
y_bmi = df_model['BMI_calc']

print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Nulls in X: {X.isnull().sum().sum()}')
print()
print('Calories_Burned:', f'min={y_cal.min():.0f} | max={y_cal.max():.0f} | mean={y_cal.mean():.0f}')
print('BMI_calc:',        f'min={y_bmi.min():.1f} | max={y_bmi.max():.1f} | mean={y_bmi.mean():.1f}')

### 3.1 Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Feature Distributions', fontsize=13, fontweight='bold')

colors = ['#4C9BE8','#2A9D8F','#F4A261','#E76F51','#A855F7',
          '#EC4899','#10B981','#F59E0B','#6366F1']

for ax, feat, color in zip(axes.flat, deploy_features, colors):
    ax.hist(df_model[feat], bins=30, color=color, alpha=0.8, edgecolor='white', linewidth=0.4)
    ax.set_title(feat.split(' (')[0], fontsize=9, fontweight='bold')
    ax.set_ylabel('Count')

axes[1][4].axis('off')
plt.tight_layout()
plt.show()

### 3.2 Feature Correlation with Targets

In [ ]:
corr_cal = X.corrwith(y_cal).sort_values(ascending=False)
corr_bmi = X.corrwith(y_bmi).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Correlations with Targets', fontsize=13, fontweight='bold')

colors_cal = ['#4C9BE8' if v > 0 else '#E76F51' for v in corr_cal.values]
colors_bmi = ['#2A9D8F' if v > 0 else '#E76F51' for v in corr_bmi.values]

axes[0].barh(corr_cal.index, corr_cal.values, color=colors_cal)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Calories_Burned', fontweight='bold')
axes[0].set_xlabel('Correlation')

axes[1].barh(corr_bmi.index, corr_bmi.values, color=colors_bmi)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('BMI_calc', fontweight='bold')
axes[1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
scaler_cal = StandardScaler()
scaler_bmi = StandardScaler()

X_cal_scaled = scaler_cal.fit_transform(X)
X_bmi_scaled = scaler_bmi.fit_transform(X)

Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X_cal_scaled, y_cal, test_size=0.2, random_state=42)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(
    X_bmi_scaled, y_bmi, test_size=0.2, random_state=42)

print(f'Train: {len(Xtr_c)} | Test: {len(Xte_c)}')

## 5. Train All Models — Calories_Burned

In [ ]:
models_cal = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

results_cal = []
trained_cal = {}

for name, m in models_cal.items():
    m.fit(Xtr_c, ytr_c)
    p    = m.predict(Xte_c)
    rmse = np.sqrt(mean_squared_error(yte_c, p))
    mae  = mean_absolute_error(yte_c, p)
    r2   = r2_score(yte_c, p)
    trained_cal[name] = m
    results_cal.append({'Model': name, 'R²': round(r2,4),
                        'RMSE': round(rmse,2), 'MAE': round(mae,2)})
    print(f'✅ {name}: R²={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}')

results_cal_df = pd.DataFrame(results_cal).sort_values('R²', ascending=False)
print()
print(results_cal_df.to_string(index=False))

## 6. Train BMI Model — Linear Regression

In [ ]:
lr_bmi = LinearRegression()
lr_bmi.fit(Xtr_b, ytr_b)
p_bmi = lr_bmi.predict(Xte_b)

r2_bmi   = r2_score(yte_b, p_bmi)
rmse_bmi = np.sqrt(mean_squared_error(yte_b, p_bmi))
mae_bmi  = mean_absolute_error(yte_b, p_bmi)

print(f'BMI Linear Regression:')
print(f'  R²   = {r2_bmi:.4f}')
print(f'  RMSE = {rmse_bmi:.3f} kg/m²')
print(f'  MAE  = {mae_bmi:.3f} kg/m²')
print()
print('Note: BMI predicted WITHOUT using Weight or Height')
print('Only activity & nutrition features used — proving real signal exists')

## 7. Model Comparison — Calories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Comparison — Calories_Burned', fontsize=13, fontweight='bold')

metrics  = ['R²', 'RMSE', 'MAE']
colors   = ['#4C9BE8', '#2A9D8F', '#F4A261']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results_cal_df['Model'], results_cal_df[metric],
                  color=color, alpha=0.85, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticklabels(results_cal_df['Model'], rotation=15, ha='right')
    for bar, val in zip(bars, results_cal_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + results_cal_df[metric].max()*0.02,
                str(val), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Actual vs Predicted Plots

In [ ]:
best_cal_name  = results_cal_df.iloc[0]['Model']
best_cal_model = trained_cal[best_cal_name]
y_pred_best    = best_cal_model.predict(Xte_c)
y_pred_lr      = trained_cal['Linear Regression'].predict(Xte_c)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Actual vs Predicted — Calories_Burned', fontsize=13, fontweight='bold')

for ax, y_pred, label, color in zip(
    axes,
    [y_pred_lr, y_pred_best],
    ['Linear Regression', best_cal_name],
    ['steelblue', '#F4A261']
):
    ax.scatter(yte_c, y_pred, alpha=0.35, color=color, s=15, edgecolors='white', linewidths=0.3)
    mn = min(float(yte_c.min()), y_pred.min())
    mx = max(float(yte_c.max()), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect fit')
    r2 = r2_score(yte_c, y_pred)
    ax.text(0.05, 0.92, f'R² = {r2:.3f}', transform=ax.transAxes,
            fontsize=11, color='darkred', fontweight='bold')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Actual (kcal)')
    ax.set_ylabel('Predicted (kcal)')
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Plots — Are Errors Random?', fontsize=13, fontweight='bold')

for ax, y_pred, label, color in zip(
    axes,
    [y_pred_lr, y_pred_best],
    ['Linear Regression', best_cal_name],
    ['steelblue', '#F4A261']
):
    residuals = np.array(yte_c) - y_pred
    ax.scatter(y_pred, residuals, alpha=0.3, color=color, s=15)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Predicted Value')
    ax.set_ylabel('Residual (Actual - Predicted)')
    ax.text(0.05, 0.93, 'Random cloud = honest model',
            transform=ax.transAxes, fontsize=9, color=color, style='italic')

plt.tight_layout()
plt.show()

## 10. Feature Importance — Best Model

In [ ]:
if hasattr(best_cal_model, 'feature_importances_'):
    imp = pd.Series(best_cal_model.feature_importances_, index=deploy_features)
    imp.sort_values(ascending=True).plot(
        kind='barh', figsize=(10, 5), color='#F4A261', edgecolor='white'
    )
    plt.title(f'Feature Importance — {best_cal_name}', fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()
else:
    coef = pd.Series(np.abs(best_cal_model.coef_), index=deploy_features)
    coef.sort_values(ascending=True).plot(
        kind='barh', figsize=(10, 5), color='steelblue', edgecolor='white'
    )
    plt.title('Feature Coefficients — Linear Regression (Absolute)', fontweight='bold')
    plt.xlabel('|Coefficient|')
    plt.tight_layout()
    plt.show()

## 11. Export Models

In [ ]:
import joblib, os

os.makedirs('models/wael_lr', exist_ok=True)

# Save best calorie model + Linear Regression for comparison
joblib.dump(trained_cal['Linear Regression'], 'models/wael_lr/lr_cal.pkl')
joblib.dump(best_cal_model,                   'models/wael_lr/best_cal_model.pkl')
joblib.dump(lr_bmi,                           'models/wael_lr/lr_bmi.pkl')
joblib.dump(scaler_cal,                       'models/wael_lr/scaler_cal.pkl')
joblib.dump(scaler_bmi,                       'models/wael_lr/scaler_bmi.pkl')

# Save best model name for Streamlit
import json
with open('models/wael_lr/best_model_name.json', 'w') as f:
    json.dump({'name': best_cal_name}, f)

print(f'✅ Best model: {best_cal_name}')
print(f'✅ Calories R²: {results_cal_df.iloc[0]["R²"]}')
print(f'✅ BMI R²: {r2_bmi:.4f}')
print(f'✅ Saved to models/wael_lr/')